<font size="+4">`Signal and Audio Processing`</font>

<font size="+3">`Seminar 09: Intro to Generative Text-to-Speech`</font>

<font size="+2">`Maks Nakhodnov & Dmitry Kropotov`</font>

<font size="+2">`Bremen, 2026`</font>

What you will learn from this notebook:

* **TTS problem formulation:** the *one-to-many* nature of speech generation and how to evaluate synthesized speech.
* **Modern TTS approaches:** from classical mel-spectrogram-based AR/NAR models to discrete audio tokens and LLM-based architectures.

In [1]:
import io
import os

os.environ['TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL'] = '1'

from urllib.request import urlopen

import regex

import numpy as np

import torch
import torchaudio
import torchmetrics
import torch.nn.functional as F
from speechbrain.pretrained import EncoderClassifier
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

import soundfile

import matplotlib
import matplotlib.pyplot as plt

from IPython.display import Audio

import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

/tmp/ipykernel_35944/26956645.py:16: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier


# `1. TTS Task Formulation and the One-to-Many Problem`

**Text-to-Speech (TTS)** is the task of generating human speech from text. While ASR is a *recognition* task (compressing a continuous noisy signal into discrete semantic tokens), TTS is a **generation** task requiring the reconstruction of lost information.

Let $Y = (y_1, y_2, \dots, y_N)$ be the input text sequence, where $y_n \in V$ (letters, phonemes, or tokens). 
The TTS task is to generate an audio signal $S = (\mathbf{s}_1, \dots, \mathbf{s}_T)$ or a sequence of acoustic features $X = (\mathbf{x}_1, \dots, \mathbf{x}_M)$, maximizing the probability:

$$ X^* = \arg\max_{X} P(X \mid Y, \mathcal{C}) $$

Where $\mathcal{C}$ is a contextual condition (speaker identity, emotion, speed, room acoustics).

## `1.1. The One-to-Many Problem`

The main fundamental difficulty of TTS is that the mapping of text to sound is **not injective**. The exact same phrase "Hello, how are you?" can be pronounced in an infinite number of ways:
* **Pitch, Timbre:** With different timbres (man, woman, child).
* **Prosody, Speed:** With varying speeds and pauses.
* **Energy, Emotions:** With different intonations (sarcasm, joy, sadness).
* **Environment:** In different environments, with varying background noise.

<b style='color:red;'>Why can't we simply compute the L1/MSE Loss between the predicted and reference spectrogram in TTS, as is done, for example, in image generation?</b>

<details><summary>Answer:</summary>>> 
    
1. Time alignment problem: Speech generated slightly faster or slower than the reference will yield a large MSE loss, even if it sounds perfect to a human. <br>
2. MSE penalizes the model for choosing a different, yet valid intonation. The model learns to predict the mathematical expectation of all spectrograms for a given text, which physically looks like a blurred spectrogram without clear harmonics. This *Oversmoothing* effect leads to the generation of a boring, robotic voice.
</details>

## `1.2. Structure of the TTS Pipeline`

<img src="ASR_vs_TTS.svg" alt="" >

There are many options for choosing each component in the TTS pipeline:

1. **Pre-processing:**
    * Text $\to$ Normalization $\to$ Phonemes $\to$ Prosody extraction.
    * Text $\to$ BPE tokens \& Audio reference $\to$ Discrete audio tokens.
    * ...
2. **Acoustic Model:**
    * Phonemes $\to$ Mel-spectrogram (*Tacotron 2*, *FastSpeech 2* models).
    * Joint modeling of semantics and acoustics (LLMs or Flow models).
    * ...
3. **Acoustic Features:**
    * Mel/Linear-spectrogram
    * Latent hidden representations
    * Discrete audio codes
    * ...
5. **Vocoder:**
    * Mel-spectrogram $\to$ Waveform (*HiFi-GAN*, *Vocos* models).
    * Decoding tokens or latents back into raw audio.
    * ...

Whatever the intermediate representation (spectrogram or tokens), the final step is always conversion into a raw sound wave — a one-dimensional array of amplitudes with a sampling rate of $24\,000 - 48\,000$ samples per second. This is handled by the **Vocoder**.

<b style='color:red;'>Why can't we discard the vocoder and force the acoustic model to predict the Raw Waveform directly (e.g., autoregressively)?</b>

<details><summary>Answer:</summary>>> 
Because of the sampling rate. For 1 second of audio at 24 kHz, the model needs to make 24,000 autoregressive steps. WaveNet did exactly this, but its inference took minutes for a one-second fragment. Vocoders solve the "resolution" problem: the acoustic model operates at a low frequency (\~50 Hz), predicting meaning and melody, while the vocoder parallelly generates high-frequency acoustic details up to 24 kHz.
</details>

## `1.3. Text Frontend: Normalization and G2P`

Before the acoustic model starts generating audio, the raw text must be prepared. Although modern LLM-based approaches can use language structure information for E2E text preprocessing, high-quality preprocessing is still important for real-world systems.

### `Text Normalization`

TTS models cannot read numbers, symbols, and abbreviations directly. Normalization translates non-alphabetic characters into their spoken (orthographic) representation:

| Raw text | Context | Normalized text (how to read it) |
| :--- | :--- | :--- |
| `1984` | In **1984**, the novel was published... | *nineteen eighty-four* |
| `1984` | My pin-code: **1984** | *one nine eight four* |
| `St.` | **St.** Patrick's Day | *Saint* |
| `St.` | Wall **St.** | *Street* |
| `$120.5` | Price: **$120.5** | *one hundred twenty dollars and fifty cents* |

**It is important to note that this task is non-trivial, as the result of disambiguation depends on the context!.**

Historically, manually written rule sets by linguists were used to solve this task. Today, neural models or LLMs operating in a translation mode from raw text to normalized text are applied.

<b style='color:red;'>What is the danger of using neural networks for text normalization?</b>

<details><summary>Answer:</summary>>> 
    
<b>Hallucinations and word omissions.</b> If manual rules do not know a word, it will simply be spelled out. An LLM, on the other hand, might rephrase the text, drop an "inconvenient" number, or replace it with a random one. Therefore, hybrid Rule-based + Neural approaches still prevail in the industry.

</details>

### `Grapheme-to-Phoneme (G2P)`

Letters (graphemes) do not always unambiguously reflect the pronunciation of a word. In English, the same combination of letters can be read in dozens of ways (e.g., *read* $\to$ /riːd/ or /rɛd/). In Russian, there is vowel reduction (корова $\to$[к а р о́ в а]) and **homographs** (words that are spelled the same but sound different due to stress):

* I opened the door **lock** (замóк).
* We went on an excursion to the old **castle** (зáмок).

G2P models assign stress and translate normalized text into transcription.

<b style='color:red;'>Why translate text into phonemes at all, if modern Audio LLMs can directly work with text BPE tokens?</b>

<details><summary>Answer:</summary>>> 
    
Using BPE indeed simplifies the pipeline, and models learn to implicitly associate tokens with sounds. However:
1. BPE splits words logically, not phonetically. Rare words, proper nouns, or new loanwords will be read by the model with stress or pronunciation errors.
2. Phonemes make the model more stable and allow manual correction of pronunciation errors in dictionaries without retraining the entire acoustic model. Nevertheless, SOTA models increasingly tend to abandon explicit G2P when scaling data.

</details>

More discussion on this topic [here](https://huggingface.co/blog/hexgrad/g2p).

## `1.4. Evaluation Methods and Metrics`

### `Performance metrics`

Evaluating the speed of a TTS pipeline follows a similar path as for ASR:

1. **RTF (Real-Time Factor):**
  
   This is the ratio of the time spent processing the audio to the duration of the audio itself:

   $$\text{RTF} = \frac{\text{Processing Time}}{\text{Audio Duration}}$$
    Modern Flow Matching models have $\text{RTF} \approx 0.05$.

    Similar to ASR, the inverse value is sometimes considered: $\text{RTFx} = \frac{1}{\text{RTF}}$

2. **First Chunk Latency:**

   Just as the time to the first token was important for Streaming ASR, for TTS, the time to the first generated chunk is measured.

<img src="https://pic3.zhimg.com/v2-29920fb92ba21fdc5fcdb6270e1e141c_1440w.jpg" alt="" >

### `Objective quality metrics`

In generation tasks, there is no single mathematical quality metric, since "naturalness" is subjective. Historically, acoustic metrics were used, but today the industry has shifted to hybrid and LLM-based approaches.

1. **Word Error Rate / CER:**

   Measures *Intelligibility* and stability. The generated audio is recognized through an ASR pipeline using strong models, usually *Whisper*. If the generative model "swallows" letters, loops, or hallucinates, the WER will increase.

<b style='color:red;'>Why will such an approach work? Won't ASR add its own recognition errors?</b>

<details><summary>Answer:</summary>>> 

Usually, TTS generation provides studio-quality audio. For the domain of high-quality audio without background noise, the error of modern ASR models is minimal.
 
</details>
   
2. **Speaker Similarity (SS):**

   Measures the quality of *Zero-Shot Voice Cloning*. Speaker embeddings from the original reference audio and from the generated audio are extracted using a pre-trained model. 
   $$ \text{SIM} = \cos(\mathbf{e}_{\text{ref}}, \mathbf{e}_{\text{gen}}) = \frac{\mathbf{e}_{\text{ref}} \cdot \mathbf{e}_{\text{gen}}}{\|\mathbf{e}_{\text{ref}}\| \|\mathbf{e}_{\text{gen}}\|} $$
   Values of $\text{SIM} > 0.7$ usually mean that a human cannot tell the voices apart.

| Model | Text token | Speech Token | WER (%) | #INS+DEL | #SUB | SS (%) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Original | - | - | 3.01 | 66 | 200 | 69.67 |
| VALL-E (Wang et al., 2023) | Phone | Encodec | 18.70 | 342 | 1312 | 53.19 |
| UniAudio (Yang et al., 2023) | Phone | Encodec | 8.74 | 254 | 519 | 47.56 |
| SpearTTS (Kharitonov et al., 2023) | Phone | Hubert | 6.14 | 133 | 410 | 51.71 |
| Exp-1-LibriTTS | Phone | Hubert | 7.41 | 325 | 409 | 67.85 |
| Exp-2-LibriTTS | Phone | S<sup>3</sup><sub>en</sub> | 5.05 | 122 | 325 | 67.85 |
| Exp-3-LibriTTS | BPE<sub>en</sub> | S<sup>3</sup><sub>en</sub> | 3.93 | 108 | 239 | 67.85 |
| Exp-4-LibriTTS | BPE | S<sup>3</sup> | 4.76 | 134 | 287 | 65.94 |
| Exp-4-Large-scale | BPE | S<sup>3</sup> | **3.17** | **96** | **184** | **69.49** |

In [2]:
from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector

@torch.no_grad()
def get_embedding(audio_path, feature_extractor, model):
    audio, sample_rate = soundfile.read(audio_path)
    audio = torch.tensor(audio)

    audio = torchaudio.functional.resample(
        audio, orig_freq=sample_rate, new_freq=16000
    )
    inputs = feature_extractor(
        audio.numpy(), return_tensors="pt", sampling_rate=16000
    )
    
    [embedding] = model(**inputs).embeddings
    return embedding / torch.linalg.norm(embedding)

In [3]:
model_name = "microsoft/wavlm-base-plus-sv"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = WavLMForXVector.from_pretrained(model_name).eval()

emb1 = get_embedding('./audio_test.wav', feature_extractor, model)
emb2 = get_embedding('./../../Tasks/Task 01/data/singing.wav', feature_extractor, model)
torch.dot(emb1, emb2).item()

/home/maksim64/miniconda3/envs/torch/lib/python3.12/site-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


0.2793833017349243

3.[**Fréchet DeepSpeech Distance (FDSD) / Kernel DeepSpeech Distance (KDSD):**](https://arxiv.org/pdf/1909.11646)

   An analogue of FID/KID in CV. It measures the distance between the distributions of real speech and generated speech in the feature space of a pre-trained ASR.
   $$ \text{FDSD} = \mathcal{W}_{2}(\mathcal{P}_{\text{reference}}, \mathcal{P}_{\text{source}}) \approx \mathcal{W}_{2}(\mathcal{N}(\mu_r, \Sigma_r), \mathcal{N}(\mu_s, \Sigma_s))$$

   $$
    \text{KDSD} = \text{MMD}(f_{\text{reference}}, f_{\text{source}})^{2}
   $$

### `Subjective quality metrics`

1. **MOS (Mean Opinion Score):** Human evaluation from 1 to 5 (usually, on Naturalness and Speaker Similarity parameters). Long and expensive.

| Model | MOS | FDSD | cFDSD | KDSD ×10<sup>5</sup> | cKDSD ×10<sup>5</sup> |
| :--- | :--- | :--- | :--- | :--- | :--- |
| *natural speech* | 4.55 ± 0.075 | 0.161 | N/A | 0 | 0 |
| *WaveNet*, van den Oord et al. (2016) | 4.41 ± 0.069 | | | | |
| *Parallel WaveNet*, van den Oord et al. (2018) | 4.41 ± 0.078 | | | | |
| FullD | 1.889 ± 0.057 | 4.51 | 4.46 | 785 | 782 |
| cRWD<sub>1</sub> | 3.394 ± 0.058 | 0.362 | 0.247 | 35.2 | 30.9 |
| cRWD<sub>{1,2,4,8,15}</sub> | 3.498 ± 0.059 | 0.398 | 0.284 | 42.1 | 37.9 |
| cRWD<sub>1</sub> + uRWD<sub>1</sub> | 3.502 ± 0.057 | 0.259 | 0.144 | 16.6 | 12.3 |
| (cRWD<sub>1</sub> + uRWD<sub>1</sub>)<sup>×5</sup> | 3.526 ± 0.054 | 0.194 | 0.073 | 5.59 | 1.34 |
| RWD<sub>1,240×{1,2,4,8,15}</sub> | 4.154 ± 0.050 | 0.184 | 0.061 | 3.73 | 0.54 |
| RWD<sup>*</sup><sub>480</sub> | 4.195 ± 0.045 | 0.193 | 0.069 | 5.28 | 0.98 |
| GAN-TTS (RWD<sup>*</sup>) | 4.213 ± 0.046 | 0.184 | 0.060 | 3.84 | 0.37 |

2. **ABX Test:** The listener is given audio A (model 1), audio B (model 2), and reference X. They need to choose which one is closer to X.
   
3. [**MLLM-as-a-Judge**:](https://arxiv.org/pdf/2510.14664v1)

With the growing popularity of *Instruction-guided TTS* (when the model uses complex instructions, for example: "read this sarcastically"), classical metrics have stopped working. Now, using multimodal LLMs as automatic assessors is the standard.
    The generated audio, text, and prompt instruction are loaded into the MLLM, after which it evaluates:
* **Instruction Following:** Did the model follow the complex instruction (pauses, accent, sighs)?
* **Expressiveness:** Emotional richness of the speech.

### `Datasets and SOTA Benchmarks`

In ASR, audio data must be diverse for high-quality and robust recognition. However, for TTS, the picture is the opposite — if the audio generation pronounces something other than what is fed into the model's input, TTS will learn incorrect pronunciations. Moreover, unconditional models without additional speaker information will average the speech of different speakers, leading to unnatural generation. Therefore, almost always, **clean single-speaker studio datasets** were used for older TTS models. However, modern models that allow **speaker conditioning** (speaker embedding, style tokens, semantic prompts, etc.) are effectively trained on **multi-speaker** sets. Such models are trained on hundreds of thousands of hours of speech (from podcasts, YouTube) so that the model sees the full variety of timbres and acoustic conditions.

<b style='color:red;'>ASR datasets almost always use a sampling rate of 16 kHz. Why is 24 kHz or even 48 kHz the standard for modern TTS datasets?</b>

<details><summary>Answer:</summary>>> 

1. In ASR, the main goal is to extract meaning. All the core linguistic and semantic information of human speech (formants of vowels and most consonants) lies in the frequency range up to 8 kHz. According to the Nyquist-Shannon sampling theorem, a sampling rate of 16 kHz is sufficient to reconstruct frequencies up to 8 kHz without loss. 
2. In TTS, the goal is naturalness and sound quality. Frequencies above 8 kHz contain higher overtones, breath, clear sibilants, and individual timbre coloration. If a voice is generated at 16 kHz, it will sound muffled. Studio-quality clean sound requires the generation of high frequencies, so TTS datasets and vocoders operate at 24 kHz (covering frequencies up to 12 kHz) and higher.

</details>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; text-align: left;">
  <tr style="background-color: #f2f2f2; font-weight: bold; border-bottom: 2px solid black;">
    <th style="padding: 10px;">Dataset Name</th>
    <th style="padding: 10px;">Size (Hours)</th>
    <th style="padding: 10px;">Domain / Features</th>
    <th style="padding: 10px;">Year</th>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LJSpeech</b></td>
    <td style="padding: 10px;">24</td>
    <td style="padding: 10px;">Studio recording, 1 speaker. Historical baseline for Tacotron/FastSpeech.</td>
    <td style="padding: 10px;">2017</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LibriTTS</b></td>
    <td style="padding: 10px;">585</td>
    <td style="padding: 10px;">Audiobooks, 2456 speakers. Standard for early multi-speaker models.</td>
    <td style="padding: 10px;">2019</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>GigaSpeech</b></td>
    <td style="padding: 10px;">10,000</td>
    <td style="padding: 10px;">Podcasts, YouTube. Spontaneous speech, noise. Kickstarted zero-shot models.</td>
    <td style="padding: 10px;">2021</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>Emilia</b></td>
    <td style="padding: 10px;">> 200,000</td>
    <td style="padding: 10px;">Large-scale In-the-wild datasets in different languages. High variability.</td>
    <td style="padding: 10px;">2025</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd">
    <td style="padding: 10px;"><b>SpeechCraft / Parler-TTS</b></td>
    <td style="padding: 10px;">2,000 / 50,000</td>
    <td style="padding: 10px;"><b>Description-based:</b> Audio is accompanied by a text description ("A man speaks calmly...").</td>
    <td style="padding: 10px;">2024</td>
  </tr>
</table>

### `Examples of metrics for modern models`

Evaluation of models in generation and in the model's ability to copy a voice from a 3-second reference on the LibriSpeech test-clean benchmark. More metrics can be [viewed here](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512#evaluation).

| Model | Architecture | WER $\downarrow$<br>*(Single Speaker / Zero-Shot)* | SS (Voice Similarity) $\uparrow$ | RTF $\downarrow$ |
| :--- | :--- | :---: | :---: | :---: |
| *Ground Truth* | *Human (LibriSpeech)* | *1.8 - 2.1* | *0.730* | *-* |
| [Tacotron 1](https://arxiv.org/abs/1703.10135) (2017) | RNN (Seq2Seq) + Griffin-Lim | \~4.0 / -  | low | \~0.50 - 1.0 |
| [Tacotron 2](https://arxiv.org/abs/1712.05884) (2017) | CNN + BiLSTM + Attention | \~2.5 / **>10.0** | low | > 1.0 (WaveNet)<br>\~0.10 (WaveGlow) |
| [FastSpeech 1](https://arxiv.org/abs/1905.09263) (2019) | Transformer (NAR) | \~3.0 / **>8.0** | low | ~0.03 |
|[FastSpeech 2](https://arxiv.org/abs/2006.04558) (2020) | Transformer (NAR) + Variance Adaptors| \~2.5 / **\~7.7** | low | **\~0.01 - 0.02** |
| [VALL-E](https://arxiv.org/abs/2301.02111) (2023) | AR + NAR | 5.9 | 0.580 | 0.50 |
|[VoiceBox](https://arxiv.org/abs/2306.15687) (2023) | Flow Matching | 1.9 | 0.681 | 0.64 |
| [CosyVoice](https://arxiv.org/abs/2407.05407) (2024)| LLM + Flow Matching | 3.2 | 0.695 | 0.92 |
|[F5-TTS](https://arxiv.org/abs/2410.06885) (2024) | Flow Matching | 2.0 | 0.647 | 0.15 |
|[Seed-TTS](https://arxiv.org/abs/2406.02430) (2024) | AR + DiT | 2.2 | **0.762** | 0.13 |

# `2. Audio Representation. Sound Tokenization.`

For a long time, the de facto standard for working with sound in neural networks has been the **mel-spectrogram**. Although this representation preserves almost all physical information about the sound, it is **continuous**, which creates a number of critical problems when trying to train truly generative models:

1.  **Regression vs Classification:** Working with a spectrogram imposes a regression task on the model — trying to predict exact amplitude values. However, speech is highly variable: the same phrase can be pronounced with different intonations. By trying to minimize the error between all possible variants, the model inevitably tends toward the mean. This leads to **spectral blurring**, causing the synthesized voice to sound robotic.
2.  **Error accumulation and instability:** In autoregressive models, predicting each subsequent spectrogram frame depends on the previous ones. In a continuous space, errors on previous frames accumulate, and the spectrogram degrades into a physically impossible signal. Discrete representations possess a self-correction property: at each step, the model must choose a specific token from a dictionary.
3.  **Lack of semantic abstractions:** A spectrogram describes the physics of the process, but not its meaning. MSE on spectrograms correlates poorly with human perception: a physically small difference might sound like severe timbre distortion to a human. We need a representation that operates at the level of concepts (phonemes, voice features, and acoustics), abstracting away from raw sound.

The solution to these problems was the use of **neural audio codecs**. They allow shifting the task from regression to classification. Such an approach turns sound generation into an audio language modeling task, where the model operates on a finite dictionary of acoustic units, and all the work of reconstructing the physical wave is handled by a pre-trained decoder.

## `2.1. Quantization of continuous representations`

If one were to try to create a dictionary of all possible sounds, its size would exceed billions of tokens. To bypass this limitation, neural codecs use **Vector Quantization (VQ)**. 

For this, a continuous vector $\mathbf{h}$ is approximated by a vector from the dictionary: $\mathbf{h} \approx \mathbf{c} \in C$. The dictionary of vectors (Codebook) can be either trainable or obtained through a deterministic procedure.

The specific quantization method depends on the balance between **Acoustic** and **Semantic** information the model needs to maintain:
* **Acoustic tokens** are focused on the physical properties of the audio signal. These tokens are intended to encode low-level acoustic characteristics, such as pitch, volume, timbre, and so on.
* **Semantic tokens** identify higher-level meaning units. These tokens determine linguistic or contextual information, rather than just acoustic properties.

<div class="pst-scrollable-table-container"><table class="table">
<thead>
<tr class="row-odd"><th class="head"><p><strong>Aspect</strong></p></th>
<th class="head"><p><strong>Acoustic tokens</strong></p></th>
<th class="head"><p><strong>Semantic tokens</strong></p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><strong>Focus</strong></p></td>
<td><p>Low-level acoustic properties</p></td>
<td><p>High-level semantic information</p></td>
</tr>
<tr class="row-odd"><td><p><strong>Goal</strong></p></td>
<td><p>Compression and reconstruction</p></td>
<td><p>Capturing semantics</p></td>
</tr>
<tr class="row-even"><td><p><strong>Training approach</strong></p></td>
<td><p>Unsupervised / Self-supervised</p></td>
<td><p>Self-supervised</p></td>
</tr>
<tr class="row-even"><td><p><strong>Applied tasks</strong></p></td>
<td><p>Better suited for audio generation and reconstruction</p></td>
<td><p>Better suited for classification and speech understanding tasks</p></td>
</tr>
<tr class="row-even"><td><p><strong>Compression efficiency</strong></p></td>
<td><p>Usually higher</p></td>
<td><p>Can be lower, as priority is given to semantics</p></td>
</tr>
<tr class="row-odd"><td><p><strong>Robustness to noise</strong></p></td>
<td><p>Often more sensitive to acoustic variations</p></td>
<td><p>Can be more robust to minor audio variations</p></td>
</tr>
<tr class="row-even"><td><p><strong>Multilinguality</strong></p></td>
<td><p>Usually language-independent</p></td>
<td><p>May capture language-specific features</p></td>
</tr>
</tbody>
</table>
</div>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed;">
    <thead>
        <tr style="text-align: center; background-color: #f2f2f2;">
            <th style="padding: 15px; border: 1px solid #444;">
                <a href="https://arxiv.org/pdf/2210.13438">EnCodec: High Fidelity Audio Compression</a>
            </th>
            <th style="padding: 15px; border: 1px solid #444;">
                <a href="https://arxiv.org/pdf/2209.03143">AudioLM: a Language Modeling Approach to Audio Generation</a>
            </th>
            <th style="padding: 15px; border: 1px solid #444;">
                <a href="https://arxiv.org/pdf/2502.17239v1">Baichuan-Audio: A Unified Framework for End-to-End Speech Interaction</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">
                <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*Q-Wr1LsEOnt2JL6rxmpTAA.png" style="width: 100%; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">
                <img src="https://uploads-ssl.webflow.com/6696d42284cfe85e5e20165b/6696d86b9589667cc7ccb66c_63f85da8c5921e1f19fa69a5_Untitled.png" style="width: 100%; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center;">
                <img src="https://arxiv.org/html/2502.17239v1/x2.png" style="width: 100%;">
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd; background-color: #f9f9f9;">
                <b>Neural Audio Codec</b>: classic autoencoder. The goal is to pack the signal physics as compactly as possible for network transmission and exact wave reconstruction.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd; background-color: #f9f9f9;">
                <b>Hierarchical Modeling</b>: introduces a division into <b>semantic tokens</b> and <b>acoustic tokens</b>.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd; background-color: #f9f9f9;">
                <b>Aligned Multimodal LLM</b>: tokenizer is integrated into the LLM. The model learns to understand and generate speech as a single stream within a single context window.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Uses <b>RVQ</b> — multi-layer quantization. Trained via <i>Reconstruction Loss</i> and <i>GAN</i>. Tokens store acoustics but contain almost no explicit semantics.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Discretization of hidden layers of SSL models (<b>w2v-BERT</b>, <b>HuBERT</b>) via <b>K-means</b>. This yields high-level tokens that are more stable for long generation than pure acoustics.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Uses <b>Finite Scalar Quantization</b>. Trained via <b>Supervised Multi-task</b> (ASR + LID + SER). Tokens immediately carry both text and emotional charge.
            </td>
        </tr>
    </tbody>
</table>

## `2.2. Residual Vector Quantization (RVQ)`

Since compressing continuous vectors into a small discrete space inevitably leads to information loss, there is a desire to control the trade-off between memory and quality. One way to enhance the expressive power of a discrete representation is to use hierarchical coding:

**RVQ** approximates a continuous vector $\mathbf{h}$ with a sum of $N_{q}$ discrete vectors from $N_{q}$ ($\approx 1-32$) independent dictionaries of size $\approx 2^{10}$:
   * The first quantizer $C_1$ selects the token whose vector is closest to $\mathbf{h}$: $\mathbf{c}_{1} = \arg\min\limits_{\mathbf{c}_{i} \in C_{1}}||\mathbf{h} - \mathbf{c}_i||_{2}$. This token encodes the core **semantic** part of the sound.
   * The second quantizer $C_2$ encodes the residual of the first quantizer $\mathbf{r}_1 = \mathbf{h} - \mathbf{c}_{1}$, capturing **timbre and intonation**.
   * The third ($C_3$) and subsequent levels encode fine acoustic details (background noise, reverberation).

<b style='color:red;'>What is the problem with using RVQ tokens in autoregressive LLMs?</b>

<details><summary>Answer:</summary>>> 
In classic text LLMs, text is a one-dimensional sequence: 1 token per 1 time step. But in a codec, one time step corresponds to a set of $N_{q}$ tokens. 

If predicted autoregressively one by one, unfolding them into a long chain, the sequence will become $N_{q}$ times longer, and Attention computation will become impossible due to quadratic complexity. If predicted in parallel, causal relationships within the frame are violated.

To solve this problem, exact autoregression can be abandoned in favor of [Inexact autoregressive decomposition](https://arxiv.org/pdf/2306.05284):

<img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*uBeLtV1rY-IFRuXCnWHSMg.png">

</details>

# `3. Acoustic Models`

Now, let's consider how to solve the task of translating text into an audio representation. The acoustic model solves the mapping task: how to translate a linguistic representation into an acoustic one.

## `3.1. Autoregressive Models (AR) + Mel-spectrograms`

The classic approach is to use a Sequence-to-Sequence architecture to predict continuous Mel-spectrograms frame by frame:

1. **Encoder** encodes the text.
2. **Attention** tries to figure out which word is currently being spoken, establishing a *soft alignment* between the text and the generated sound.
3. **Decoder** autoregressively generates the spectrogram.

Main drawback: **Instability and slow inference**. 

<b style='color:red;'>Why do Attention-based AR models often skip words or start endlessly repeating the same phoneme?</b>

<details><summary>Answer:</summary>>> 
    
During inference, the model needs to understand when it's time to shift its focus to the next letter. If complex, long, or unusual words appear in the text, the focus will either get stuck on one letter or jump over several words. Due to the autoregressive nature, an error at one step ruins further generation.
</details>

Three features of continuous autoregressive models can be highlighted:
1. Similar to an LLM, a TTS model must predict an <eos> token to stop generation. For this, all AR models on continuous representations contain a separate Stop Token head predicting the stopping probability.
   
<b style='color:red;'>What should the loss function for the Stop Token head look like?</b>

<details><summary>Answer:</summary>>> 
    
A standard Cross-entropy loss can be used; however, one must account for the significant imbalance between the MSE/L1 loss for the spectrogram and the Stop loss. The latter usually needs to be scaled up. 
</details>

2. Most continuous-representation AR models use a special *convolutional* Post-net to improve and **smooth the spectrogram fully predicted by the AR model**.

3. Since a continuous spectrogram has a significant correlation between adjacent frames, it might be beneficial for the model to predict the spectrum of the current frame at the next step, leading to obviously incorrect generation. To prevent the model from getting stuck in such a local optimum, the use of global context must be forced. In particular, Dropout can be used both during training and inference. For older models, a Dropout of $\approx 0.5$ showed the best results. In newer models, Dropout is partially replaced by stochastic forward passes, for example, through a Latent Sampling Module. 

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/1712.05884" style="text-decoration: none; color: #0066cc;">Natural TTS Synthesis by Conditioning WaveNet on Mel Spectrogram Predictions (Tacotron 2)</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/1809.08895" style="text-decoration: none; color: #0066cc;">Neural Speech Synthesis with Transformer Network (Transformer TTS)</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2407.08551" style="text-decoration: none; color: #0066cc;">Autoregressive Speech Synthesis without Vector Quantization (MELLE)</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://ar5iv.labs.arxiv.org/html/1809.08895/assets/Tacotron2.jpg" alt="Tacotron 2" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://ar5iv.labs.arxiv.org/html/1809.08895/assets/Transformer.jpg" alt="Transformer TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://arxiv.org/html/2407.08551v2/x1.png" alt="MELLE" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Seq2Seq model based on <b>LSTM</b>. Uses <i>Location-Sensitive Attention</i> for text and audio alignment.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Seq2Seq model built entirely on <b>Multi-Head Self-Attention</b>, replicating the original Encoder-Decoder Transformer architecture.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Decoder-only LLM</b>, but operating directly with <b>continuous</b> mel-spectrogram values.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                The encoder translates text into hidden representations. At each step, the decoder looks at the previously generated spectrogram frame, selects the required phoneme via Attention, and autoregressively predicts the next frame.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                The decoder uses hidden text representations from the encoder via Cross-Attention and autoregressively outputs spectrogram frames. Attention allows linking distant contexts.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                The model solves the In-Context Learning (Voice Cloning) task. MELLE generates distribution parameters ($\mathcal{N}(\mu, \sigma)$), from which a latent vector is sampled, turning into <b>spectrogram frames</b>. It can predict several (<i>r</i>) frames at once per step.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Trained via <i>Teacher-Forcing</i>.</li>
                    <li>Optimizes L1/MSE loss of spectrograms.</li>
                    <li>Training is very slow due to recurrent LSTM layers.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Thanks to Self-Attention, gradients during the Teacher-Forcing stage are computed in parallel.</li>
                    <li>Requires a special <i>Scaled Positional Encoding</i> to align text and audio spaces.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Optimizes <b>Regression Loss</b> (L1 + MSE).</li>
                    <li><b>Spectrogram Flux Loss</b> encourages diverse spectrogram generation, preventing convergence into a local optimum.</li>
                </ul>
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> High naturalness.<br>
                <b>-</b> Attention often breaks during inference (swallowing/repetitions).<br>
                <b>-</b> Very slow autoregressive generation.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Intonation is built more globally thanks to Self-Attention.<br>
                <b>-</b> Attention often breaks during inference (swallowing/repetitions).<br>
                <b>-</b> Very slow autoregressive generation.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Inference can be accelerated by predicting <i>r</i> frames at a time.<br>
            </td>
        </tr>
    </tbody>
</table>

## `3.2. Non-Autoregressive Models (NAR) + Mel-spectrograms`

A key problem for AR models is slow inference, as well as general instability associated with error accumulation and errors in the **Soft Alignment** attention mechanism on Out-of-Domain data. Switching to **Hard Alignment** solves these problems. The general architecture can be described as follows:

1. **Encoder** encodes the text.
2. **Length Regulator** predicts the duration of each phoneme and duplicates the hidden phoneme representations proportionally to their predicted duration.
3. **NAR Decoder** generates the entire Mel-spectrogram in parallel in a single pass.

Main advantage: **Ultra-fast inference**.

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/1905.09263" style="text-decoration: none; color: #0066cc;">FastSpeech: Fast, Robust and Controllable Text to Speech</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2105.06337" style="text-decoration: none; color: #0066cc;">Grad-TTS: A Diffusion Probabilistic Model for TTS</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2309.03199" style="text-decoration: none; color: #0066cc;">Matcha-TTS: A Fast TTS Architecture with Conditional Flow Matching</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://publish-01.obsidian.md/access/93d590b5e88c06a4619cd08dc2123a4d/slp/9%20Speech%20Synthesis/attachments/ren-fastspeech1-arch.png" alt="FastSpeech" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://ar5iv.labs.arxiv.org/html/2105.06337/assets/grad-tts-new.png" alt="Grad-TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://cdn-ak.f.st-hatena.com/images/fotolife/N/Ninhydrin/20230927/20230927085934.png" alt="Matcha-TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>NAR</b> model based on Transformer. Trained to predict the Mel-spectrogram directly.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Score-based Diffusion</b>. The acoustic model predicts the distribution parameters, and the diffusion network iteratively transforms a noisy representation of the sound into a spectrogram.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Conditional Flow Matching</b>. The most modern replacement for classical diffusion based on Optimal Transport.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Phonemes are fed into a <i>Variance Adaptor</i>, which predicts the duration. Then the vector is stretched and the decoder outputs all spectrogram frames in parallel.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                The encoder translates text into $\mu$ — the mean for a normal distribution. The spectrogram is generated by gradually removing noise from this distribution towards real acoustic features.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                The model does not predict noise like diffusion, but learns a <i>vector field</i> — straight trajectories along which noise transitions into a spectrogram. Text and phoneme durations act as conditions.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Ultra-fast parallel training.</li>
                    <li>Optimizes L1/MSE loss functions for the spectrogram and duration.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Abandonment of hard MSE for the spectrogram in favor of a stochastic diffusion loss.</li>
                    <li>Requires solving an SDE during inference.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Training is reduced to vector field regression.</li>
                </ul>
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> High stability (no skipped words).<br>
                <b>+</b> Ultra-fast inference.<br>
                <b>-</b> <i>Oversmoothing</i> problem: averaging the loss makes the voice slightly "robotic" and muffled.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Solves the oversmoothing problem.<br>
                <b>-</b> Slow generation (requires 50–100 neural network passes for denoising).
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Quality is indistinguishable from diffusion (very high naturalness).<br>
                <b>+</b> Ultra-fast inference: requires only <b>2-4 steps</b> of an ODE solver.<br>
                <b>+</b> Very "lightweight" model in terms of memory.
            </td>
        </tr>
    </tbody>
</table>

### `Training Length Regulator and the alignment problem`

The main question in NAR models is how to train the **Length Regulator**. To stretch phoneme representations, we need to know the exact duration of each phoneme in spectrogram frames. But raw datasets usually only contain text and audio.

<b style='color:red;'>Why can't we use Soft Attention inside a NAR model so it "smears" the attention focus over the spectrogram on its own without an explicit Length Regulator?</b>

<details><summary>Answer:</summary>>> 
    
1. **Loss of temporal structure:** Soft Attention assigns probabilities. A spectrogram frame might consist of 40% sound "A" and 60% sound "B". This will lead to generating a mixture of sounds. To properly duplicate tokens over time, exact boundaries must be predicted.
2. **Violation of monotonicity:** Standard Attention is unconstrained. It can look at the end of the sentence first, then at the beginning. In speech, phonemes always follow each other strictly sequentially. 
</details>

* **Distillation from an AR teacher:**
   
Since Attention-based AR models learn Soft Alignment on their own, it can be used to generate Hard Alignment. In the FastSpeech 1 model, it was proposed to train an AR model. The most probable path is extracted from its attention matrix and is used as Ground Truth durations for training the NAR model.
* **External tools (Forced Alignment):**

Obviously, training an AR model requires significant extra resources. Therefore, classical methods can be used to determine phoneme boundaries. For example, FastSpeech 2 used **MFA (Montreal Forced Aligner)**. This is a classical system based on Hidden Markov Models that finds phoneme boundaries with high precision.
   * *Pros:* Very stable, works out of the box.
   * *Cons:* Complicates the pipeline. MFA is a separate system requiring its own training, pronunciation dictionaries, and tuning.

* **End-to-End alignment (MAS):**
  
Starting with Glow-TTS, VITS, and modern diffusion networks (Matcha-TTS), the industry shifted to training alignment without external tools directly during model training. The **Monotonic Alignment Search (MAS)** algorithm is used for this.
   The model computes a probability matrix of each audio frame corresponding to each phoneme. MAS uses dynamic programming to find the most probable *monotonic* path in this matrix. The found path is used as "hard" duration labels at the current training step.

<b style='color:red;'>What is the main problem with MAS in the early stages of training?</b>
<details><summary>Answer:</summary>>>
<b>"Cold Start" problem.</b> At the very beginning of training, the model weights are random, and the likelihood matrix is noise. MAS might find a random path, and the model will start learning from incorrect alignment. To avoid this, the model is first trained on short phrases, where it's harder to make mistakes.
</details>

![](https://open-speech-ekstep.github.io/img/glow_training_and_inference.png)

### `Controllability and Variance Adaptors`

In NAR models, the basic mechanism for controlling speech characteristics are Variance Adaptors, proposed in the FastSpeech 2 architecture. These modules consist of predictors that forecast duration (Duration), fundamental frequency (Pitch / F0), and energy (Energy) for each phoneme based on hidden text representations. At the inference stage, scalar multiplication of these predictors' outputs by given coefficients allows controlling the speed and pitch of the generated speech.

We discussed that the absence of speech characteristics during training leads to the oversmoothing effect: Pitch and Energy, implicitly learned by the model, converge to average values, causing the synthesized speech to sound monotonous and unnatural.

This problem can be solved by adding blocks that predict its Pitch, Energy, and other properties in addition to phoneme duration. This helps untangle the pronunciation of different phonemes, and also adds simple per-phoneme generation control. 

<b style='color:red;'>Where do we get training data for the predictor blocks?</b>

<details><summary>Answer:</summary>>> 

There are both classical and neural network methods for F0 prediction. These predictions serve as an explicit target during training. For energy, it is sufficient to take the norm of the corresponding frame after STFT.
    
</details>

![](https://www.chengfeng.wiki/images/fs2.png)

To control the overall speech style, arbitrary embeddings identifying a specific speaker can be added during the training process. For example, the [DiffStyleTTS](https://arxiv.org/pdf/2412.03388) model proposes a hierarchical division of control factors into three levels:
1.  **Speaker Identity (Timbre):** Speaker identity. In this model, it is implemented through a fixed-size embedding table for speakers from the training set.
2.  **Implicit Style (GST):** Global stylistic characteristics (emotional tone, acoustic conditions). Extracted from a reference audio signal by a trainable network.
3.  **Explicit Prosody (Pitch, Energy, Duration):** Explicit physical parameters. Predicted explicitly by a model conditioned on text and style vectors.

<b style='color:red;'>Why in the DiffStyleTTS architecture is the loss function calculated for Pitch and Energy, but not applied directly to the Speaker Embedding or Implicit Style Condition?</b>

<details><summary>Answer:</summary>>> 
    
Pitch, Energy, and Duration act as target variables that the acoustic model must be able to synthesize at the inference stage, given only text as input. Speaker ID and implicit style are input conditions provided externally (via reference audio or identifier). Direct optimization of the conditions themselves is not required; their correctness and representativeness are evaluated indirectly through the reconstruction loss function of the final mel-spectrogram.
</details>

<b style='color:red;'>Does this architecture have the capability for Zero-Shot synthesis: generating speech with the characteristics of a speaker absent from the training set?</b>

<details><summary>Answer:</summary>>> 
    
Resolving this question requires separating the characteristics:
1. <b>Zero-Shot prosody transfer (Style Transfer) — is possible.</b> The GST module is capable of extracting implicit style from arbitrary OOD (Out-of-Domain) reference audio and transferring the intonation contour to the target text.
2. <b>Zero-Shot timbre cloning (Voice Cloning) — is impossible.</b> Modeling speaker identity via a <i>Lookup Table</i> strictly limits generation to timbres from the training set. 

To implement full-fledged Zero-Shot cloning, the Lookup Table must be replaced with a pre-trained discriminative speaker encoder (e.g., based on WavLM, ECAPA-TDNN architectures), capable of extracting an invariant timbre vector from an arbitrary audio signal.
</details>

![](https://xuan3986.github.io/DiffStyleTTS/demos/DiffStyleTTS.png)

## `3.3. Autoregressive Models (AR) + Discrete tokens`

The emergence of neural audio codecs (EnCodec, SoundStream) made it possible to compress a continuous wave into a sequence of discrete tokens: **TTS has turned into a language modeling task**. Now, one can feed the model a text prompt, an audio reference in the form of tokens, and target text. The model will autoregressively continue the sequence of tokens, relying on the context, thereby copying the voice, emotion, and even room acoustics (Zero-Shot In-Context Learning).

Instead of training on clean studio data, these models are trained on tens of thousands of hours of noisy data (podcasts, audiobooks, YouTube). Training is reduced to predicting the next token given the already generated (or input) tokens.

Main advantage: **Zero-Shot Voice Cloning**. 
The model is given a text prompt, a short voice fragment of the target speaker (usually ~3 seconds) as audio tokens, and target text. The model autoregressively continues the sequence of audio tokens, copying the timbre, room acoustics, and speaking manner from the prompt without fine-tuning for the new speaker.

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2301.02111" style="text-decoration: none; color: #0066cc;">Neural Codec Language Models are Zero-Shot TTS Synthesizers (VALL-E)</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2403.16973" style="text-decoration: none; color: #0066cc;">VoiceCraft: Zero-Shot Speech Editing and TTS in the Wild</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2402.08093" style="text-decoration: none; color: #0066cc;">BASE TTS: Lessons from building a billion-parameter TTS model</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://www.zauberware.com/assets/header/Screenshot-2023-01-11-at-22.29.37.png" alt="VALL-E" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/0*7cUA7yB6WSif34gi" alt="VoiceCraft" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://scx1.b-cdn.net/csz/news/800a/2024/amazon-unveils-largest.jpg" alt="BASE TTS" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                First TTS LLM model. Consists of two stages: <b>AR</b> (for the 1st RVQ layer) + <b>NAR</b> (for layers 2-8).
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Unified Transformer Decoder <b>AR</b> model using the <i>Delayed Stacking</i> and <i>Causal Masking</i> mechanism.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>AR</b> model with 1 billion parameters. Trained on 100K hours of data. Rejection of RVQ in favor of single-stream BPE tokens.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Training is reduced to language modeling. The model learns to link text BPE tokens and discrete EnCodec tokens. Perfectly copies prompt acoustics.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Allows not only continuing speech (TTS), but also <b>editing</b> it. The model masks a word within a phrase and generates a replacement considering both left and right audio context.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Instead of raw audio tokens, uses hidden states of WavLM, passed through vector quantization and Byte-Pair Encoding (BPE). This radically reduces sequence length.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> One of the first models with out-of-the-box Zero-Shot cloning.<br>
                <b>-</b> Hallucinations (pronunciation errors) are present due to weak text-audio alignment in AR.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> High quality on the speech editing task.<br>
                <b>-</b> Inference is quite resource-intensive and complex due to token permutation.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Demonstration of <i>emergent properties</i>: the model begins to correctly intonate complex syntactic structures and convey emotions, relying solely on text context.<br>
                <b>-</b> Requires colossal resources.
            </td>
        </tr>
    </tbody>
</table>

## `3.4. Non-Autoregressive Models (NAR) + Tokens / Latent representations`

The integration of discrete tokens and continuous latent representations with non-autoregressive (NAR) generation aims to solve the computational complexity problem of AR models while maintaining high naturalness and Zero-Shot capabilities. Instead of sequentially predicting the next token, these architectures generate the entire sequence in parallel. For this, iterative refinement methods are used: **Masked Audio Modeling** or **Diffusion / Flow Matching**.

<b style='color:red;'>Why is an intermediate representation in the form of semantic tokens extracted from ASR models needed, abandoning the direct "Text $\to$ Acoustics" mapping?</b>

<details><summary>Answer:</summary>>> 
    
The reason lies in the **Information Gap** problem. A direct mapping from text to an acoustic representation (which contains not only pronunciation but also timbre, background noise, micro-intonations) is too complex a task for NAR models, which often leads to hallucinations and loss of alignment.
<br><br>
Semantic tokens act as a strict information bridge: they have a rigid connection to linguistic meaning and basic prosody, but are invariant to timbre and acoustic noises. Using a cascaded pipeline <code>Text $\to$ Semantic $\to$ Acoustic</code> stabilizes the generation process, as each subnetwork solves a highly specialized task.
</details>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; table-layout: fixed; text-align: left;">
    <thead>
        <tr style="background-color: #f2f2f2; border-bottom: 2px solid #444;">
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2305.09636" style="text-decoration: none; color: #0066cc;">SoundStorm: Efficient Parallel Audio Generation</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2403.03100" style="text-decoration: none; color: #0066cc;">NaturalSpeech 3: Zero-Shot TTS with Factorized Codec and Diffusion</a>
            </th>
            <th style="padding: 15px; width: 33.33%; border: 1px solid #ddd; text-align: center;">
                <a href="https://arxiv.org/pdf/2407.08551" style="text-decoration: none; color: #0066cc;">CosyVoice: A Scalable Multilingual Zero-shot TTS Synthesizer</a>
            </th>
        </tr>
    </thead>
    <tbody>
        <tr style="background-color: white;">
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEjI75okNCRdLNSwU31uccP6wuIXVDxhhhvTI9f_V0ntiAxUOtw29TZeAvNFZXNBta9GraHB1bikM6zGQrFM7zSt2d985P6OC9On0xDwrLfj9OzJjdJ8BjrcYiAj_VGi3VQsqCIbNx_B7DP75KCU9D6xB_ybgHDpBk0z-eBkXXRdu9u04zcdzNCKGJuqszrR/s16000/image2.png" alt="SoundStorm" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/0*LqcJdrKlt1T8W9S3.jpg" alt="NaturalSpeech 3" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; vertical-align: middle;">
                <img src="https://arxiv.org/html/2407.05407v2/x1.png" alt="CosyVoice" style="max-width: 100%; max-height: 250px; object-fit: contain;">
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>NAR</b> architecture on discrete tokens. Uses the <i>Masked Audio Modeling</i> concept.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>NAR</b> architecture in a continuous latent representation space. Based on <i>Latent Diffusion</i> and strict attribute <b>Factorization</b>.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>Hybrid architecture</b>: AR model for generating semantic tokens + NAR model (<i>Conditional Flow Matching</i>) for spectrogram synthesis.
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                Relies on semantic tokens as a condition. The model itself replaces the slow AR part, generating RVQ token layers in parallel. It starts with a fully masked sequence and iteratively predicts tokens.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                The complex speech signal is broken down into independent subspaces: <i>Content, Prosody, Timbre, Acoustic Details</i>. An independent diffusion network is trained for each attribute.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                Uses Supervised Semantic Tokens from the SenseVoice ASR model. An LLM predicts semantics based on text, and a Flow Matching model deterministically translates noise into a spectrogram, relying on semantic tokens and a speaker embedding.
            </td>
        </tr>
        <tr style="vertical-align: top; background-color: #f9f9f9;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Audio synthesis is 2 orders of magnitude faster than in AR codec models.</li>
                    <li>Completely solves the looping and skipping problem.</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>High level of modular control (ability to combine one speaker's timbre with another's prosody).</li>
                </ul>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <ul>
                    <li>Flow-matching is significantly faster than classical diffusion.</li>
                </ul>
            </td>
        </tr>
        <tr style="vertical-align: top;">
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Supreme performance (generates 30 seconds of audio in 0.5 sec).<br>
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> Solves the entangled features problem.<br>
                <b>-</b> Extremely complex training and audio decomposition pipeline.
            </td>
            <td style="padding: 12px; border: 1px solid #ddd;">
                <b>+</b> High quality Zero-Shot Voice Cloning (including cross-lingual transfer).<br>
                <b>-</b> The presence of an AR module for semantics leaves a theoretical risk of hallucinations.
            </td>
        </tr>
    </tbody>
</table>

## `3.5. Generation Example`

In [ ]:
def get_speaker_embedding(audio_path):
    signal, fs = soundfile.read(audio_path)
    signal = torch.tensor(signal)
    signal = torchaudio.functional.resample(signal, orig_freq=fs, new_freq=16000)
    
    with torch.no_grad():
        embeddings = speaker_model.encode_batch(signal)
        embeddings = F.normalize(embeddings, dim=-1).squeeze(0)
    return embeddings.to(device)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)
speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb", 
    run_opts={"device": device}, 
    savedir="tmp_spkrec"
)

In [ ]:
reference_audio_path = "./audio_test.wav" 
text = "Hello, students! Today, we are learning about zero-shot voice cloning."

with torch.no_grad():
    inputs = processor(text=text, return_tensors="pt").to(device)
    speaker_embeddings = get_speaker_embedding(reference_audio_path)
    speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

Audio(speech.cpu().numpy(), rate=16000)